# Boxing punch detection: training

Trains `BoxingDetector` (RT-DETRv2 with a timm MobileNet/EfficientNet backbone) on the processed dataset.
Build the dataset first (see README):

```
uv run python -m boxing.preprocess data/raw/punch data/processed/punch
```

In [ ]:
# Colab bootstrap: clone the repo, install the boxing package, and run from notebooks/.
# Skipped when running locally.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jmanuelc87/boxing.git"
REPO = Path("/content/boxing")

if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
    if (REPO / ".git").exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    # Keep Colab's preinstalled torch/torchvision/opencv: install the project without
    # its dependencies, then add only the packages Colab does not already provide.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "timm", "transformers", "wandb", "python-dotenv", "scipy", "tqdm"],
        check=True,
    )
    os.chdir(REPO / "notebooks")
    print("colab: repo ready at", REPO)
else:
    print("local run: using the current checkout")


In [ ]:
import json, random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from dotenv import load_dotenv

from boxing.model import build_model

# Project root: the notebook may run from notebooks/ or from the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Environment variables from a .env file, Colab-style.
# In Colab the notebook runs on a fresh VM, so mount Google Drive and read .env from there.
# Locally it reads the project .env. Put the file at DOTENV_PATH in Drive (or change it).
import os
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DOTENV_PATH = Path("/content/drive/MyDrive/boxing/.env")
else:
    DOTENV_PATH = ROOT / ".env"

loaded = load_dotenv(DOTENV_PATH, override=False)
print("env:", "Colab" if IN_COLAB else "local", "| .env loaded:", loaded, "|", DOTENV_PATH)

In [ ]:
# Config
DATA_DIR = ROOT / "data/processed/punch"   # full dataset; use punch-test for a quick check
OUT_DIR = ROOT / "outputs/boxing-detector"
BACKBONE = "efficientnet"                      # or "efficientnet"
IMAGE_SIZE = 640                            # images are resized to a square
BATCH_SIZE = 8
EPOCHS = 30
WARMUP_EPOCHS = 3                        # linear LR warmup before cosine decay
DISPLAY_THRESHOLD = 0.1                     # min score to draw a predicted box
LR = 1e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
SEED = 0
NUM_WORKERS = 0                           # 0 loads in the main process; raise on Linux if data loading is the bottleneck

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Device: macOS uses MPS if available; other platforms use CUDA if available; else CPU.
import platform

if platform.system() == "Darwin" and torch.backends.mps.is_available():
    device = torch.device("mps")
elif platform.system() != "Darwin" and torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("platform:", platform.system(), "| device:", device)

## Data

In [ ]:
# Download the dataset from Kaggle and build the processed dataset with boxing.preprocess.
# Kaggle auth: set KAGGLE_API_TOKEN, or KAGGLE_USERNAME and KAGGLE_KEY, in .env (or Colab Drive .env).
import base64
import shutil
import subprocess
import urllib.request
import zipfile

KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/piotrstefaskiue/olympic-boxing-punch-classification-video-dataset"
ZIP_PATH = ROOT / "data/raw/olympic-boxing-punch.zip"
RAW_DIR = ROOT / "data/raw/punch"
PROCESSED_DIR = ROOT / "data/processed/punch"


def download(url, dst):
    if dst.exists():
        print("already downloaded:", dst)
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url)
    token, user, key = os.environ.get("KAGGLE_API_TOKEN"), os.environ.get("KAGGLE_USERNAME"), os.environ.get("KAGGLE_KEY")
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    elif user and key:
        req.add_header("Authorization", "Basic " + base64.b64encode(f"{user}:{key}".encode()).decode())
    else:
        print("warning: no Kaggle credentials found in the environment; the request may be refused")
    with urllib.request.urlopen(req) as r, dst.open("wb") as f:
        shutil.copyfileobj(r, f)
    print("downloaded:", dst, f"({dst.stat().st_size / 1e6:.0f} MB)")


def extract(zip_path, raw_dir):
    if any(raw_dir.glob("task_*")):
        print("already extracted:", raw_dir)
        return
    raw_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(raw_dir)
    # preprocess expects task_* folders directly under raw_dir; move them up if the zip nests them
    for ann in list(raw_dir.rglob("annotations.json")):
        task = ann.parent
        if task.parent != raw_dir and not (raw_dir / task.name).exists():
            shutil.move(str(task), str(raw_dir / task.name))
    print("tasks:", len(list(raw_dir.glob("task_*"))))


download(KAGGLE_URL, ZIP_PATH)
extract(ZIP_PATH, RAW_DIR)

if (PROCESSED_DIR / "labels.json").exists():
    print("already preprocessed:", PROCESSED_DIR)
else:
    subprocess.run(
        [sys.executable, "-m", "boxing.preprocess", str(RAW_DIR), str(PROCESSED_DIR)],
        cwd=ROOT, check=True,
    )


In [ ]:
labels = json.loads((DATA_DIR / "labels.json").read_text())
id2label = {int(k): v for k, v in labels["id2label"].items()}
print(id2label)

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class PunchDataset(Dataset):
    """One item per annotated frame. Boxes are normalised (cx, cy, w, h), so resizing is safe.
    No horizontal flip: it would swap left/right hand and so the class."""

    def __init__(self, split):
        self.records = [json.loads(l) for l in (DATA_DIR / f"{split}.jsonl").read_text().splitlines() if l]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        r = self.records[i]
        img = Image.open(DATA_DIR / r["image"]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
        x = torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 255
        x = (x - MEAN) / STD
        target = {
            "class_labels": torch.tensor(r["class_labels"], dtype=torch.long),
            "boxes": torch.tensor(r["boxes"], dtype=torch.float32).reshape(-1, 4),
        }
        return x, target


def collate(batch):
    xs, ts = zip(*batch)
    return torch.stack(xs), list(ts)


train_ds, val_ds = PunchDataset("train"), PunchDataset("val")
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=NUM_WORKERS)
val_dl = DataLoader(val_ds, BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)
print(len(train_ds), "train frames,", len(val_ds), "val frames")

In [ ]:
# Class balance in the training split
counts = np.bincount([c for r in train_ds.records for c in r["class_labels"]], minlength=len(id2label))
plt.figure(figsize=(8, 3)); plt.bar([id2label[i] for i in range(len(counts))], counts)
plt.xticks(rotation=60, ha="right"); plt.tight_layout(); plt.show()

## Model

In [ ]:
model = build_model(BACKBONE, id2label=id2label, pretrained_backbone=True).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

backbone_params = [p for n, p in model.named_parameters() if "backbone" in n]
other_params = [p for n, p in model.named_parameters() if "backbone" not in n]
optimizer = torch.optim.AdamW(
    [{"params": backbone_params, "lr": BACKBONE_LR}, {"params": other_params, "lr": LR}],
    weight_decay=WEIGHT_DECAY,
)
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[WARMUP_EPOCHS])

## Train

In [ ]:
from torchvision.ops import box_iou


def cxcywh_to_xyxy(b):
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], -1)


@torch.no_grad()
def mean_ap(dl, iou_thresh=0.5, score_floor=0.05, net=None):
    """mAP@iou_thresh over classes (VOC-style, all-point interpolated AP).

    Every prediction with score above `score_floor` is a detection. For each class,
    detections are ranked by score across the whole set. A detection is a true
    positive if it matches a not-yet-matched ground-truth box of the same class in
    the same frame with IoU >= iou_thresh; otherwise it is a false positive.
    AP is the area under the interpolated precision-recall curve. mAP is the mean
    AP over the classes that appear in `dl`.
    """
    net = net or model
    net.eval()
    C = len(id2label)
    dets = {c: [] for c in range(C)}   # per class: (scores, boxes, frame ids)
    gts = {c: {} for c in range(C)}    # per class: frame id -> GT boxes (xyxy)
    frame = 0
    for x, ts in dl:
        out = net(pixel_values=x.to(device))
        probs = out.logits.sigmoid().cpu()
        pred_boxes = cxcywh_to_xyxy(out.pred_boxes.cpu())
        for p, pb, t in zip(probs, pred_boxes, ts):
            for c in range(C):
                keep = (p[:, c] > score_floor).nonzero().squeeze(1)
                if len(keep):
                    dets[c].append((p[keep, c], pb[keep], torch.full((len(keep),), frame)))
            for box, cls in zip(cxcywh_to_xyxy(t["boxes"]), t["class_labels"].tolist()):
                gts[cls].setdefault(frame, []).append(box)
            frame += 1
    for c in range(C):
        gts[c] = {f: torch.stack(b) for f, b in gts[c].items()}

    aps = {}
    for c in range(C):
        n_gt = sum(len(b) for b in gts[c].values())
        if n_gt == 0:
            continue  # class absent from this split
        if not dets[c]:
            aps[id2label[c]] = 0.0
            continue
        scores = torch.cat([d[0] for d in dets[c]])
        boxes = torch.cat([d[1] for d in dets[c]])
        frames = torch.cat([d[2] for d in dets[c]])
        order = scores.argsort(descending=True)
        matched = {f: torch.zeros(len(b), dtype=torch.bool) for f, b in gts[c].items()}
        tp = torch.zeros(len(order))
        for k, i in enumerate(order.tolist()):
            f = int(frames[i])
            if f not in gts[c]:
                continue
            ious = box_iou(boxes[i][None], gts[c][f])[0]
            ious[matched[f]] = -1.0  # each GT box can be matched once
            j = int(ious.argmax())
            if ious[j] >= iou_thresh:
                tp[k] = 1.0
                matched[f][j] = True
        tp_cum = tp.cumsum(0)
        fp_cum = (1 - tp).cumsum(0)
        recall = tp_cum / n_gt
        precision = tp_cum / (tp_cum + fp_cum)
        mrec = torch.cat([torch.zeros(1), recall, torch.ones(1)])
        mpre = torch.cat([torch.zeros(1), precision, torch.zeros(1)])
        mpre = torch.flip(torch.cummax(torch.flip(mpre, [0]), 0).values, [0])  # precision envelope
        steps = (mrec[1:] != mrec[:-1]).nonzero().squeeze(1)
        aps[id2label[c]] = float(((mrec[steps + 1] - mrec[steps]) * mpre[steps + 1]).sum())
    return (float(np.mean(list(aps.values()))) if aps else 0.0), aps


@torch.no_grad()
def mean_iou(dl, score_thresh=DISPLAY_THRESHOLD, net=None):
    """mIoU over classes.

    For each ground-truth box, take the best IoU with a predicted box of the same
    class whose score is above `score_thresh` (IoU 0 if there is none). Average
    within each class, then across the classes that appear in `dl`.
    """
    net = net or model
    net.eval()
    ious = {c: [] for c in range(len(id2label))}
    for x, ts in dl:
        out = net(pixel_values=x.to(device))
        probs = out.logits.sigmoid().cpu()
        pred_boxes = cxcywh_to_xyxy(out.pred_boxes.cpu())
        for p, pb, t in zip(probs, pred_boxes, ts):
            for gt_box, gt_cls in zip(cxcywh_to_xyxy(t["boxes"]), t["class_labels"].tolist()):
                mask = p[:, gt_cls] > score_thresh
                iou = box_iou(gt_box[None], pb[mask])[0].max().item() if mask.any() else 0.0
                ious[gt_cls].append(iou)
    per_class = {id2label[c]: float(np.mean(v)) for c, v in ious.items() if v}
    return (float(np.mean(list(per_class.values()))) if per_class else 0.0), per_class


def to_device(x, ts):
    return x.to(device), [{k: v.to(device) for k, v in t.items()} for t in ts]


def run_epoch(dl, train):
    model.train(train)
    total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, ts in tqdm(dl, leave=False, desc="train" if train else "val"):
            x, ts = to_device(x, ts)
            loss = model(pixel_values=x, labels=ts).loss
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
                optimizer.step()
            total += loss.item() * len(x); n += len(x)
    return total / n


history = {"train": [], "val": [], "map": [], "miou": []}
best = None  # (val mAP, -val loss): highest mAP wins, ties go to the lower val loss
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Weights & Biases: API key comes from WANDB_API_KEY (loaded from .env above)
import wandb

run = wandb.init(
    entity="jmanuelc87-personal",
    project="Boxing",
    name=f"{BACKBONE}-{len(train_ds)}tr-{EPOCHS}ep",
    config={
        "backbone": BACKBONE, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
        "epochs": EPOCHS, "warmup_epochs": WARMUP_EPOCHS, "lr": LR,
        "backbone_lr": BACKBONE_LR, "weight_decay": WEIGHT_DECAY, "seed": SEED,
        "train_frames": len(train_ds), "val_frames": len(val_ds),
        "data_dir": str(DATA_DIR),
    },
)
for epoch in range(EPOCHS):
    tr, va = run_epoch(train_dl, True), run_epoch(val_dl, False)
    mAP, per_class = mean_ap(val_dl)
    miou, iou_per_class = mean_iou(val_dl)
    scheduler.step()
    history["train"].append(tr); history["val"].append(va); history["map"].append(mAP); history["miou"].append(miou)
    print(f"epoch {epoch + 1:02d}  train {tr:.4f}  val {va:.4f}  val mAP {mAP:.4f}  val mIoU {miou:.4f}")
    run.log({
        "epoch": epoch + 1, "train/loss": tr, "val/loss": va,
        "val/mAP50": mAP, "val/mIoU": miou, "lr": optimizer.param_groups[1]["lr"],
        **{f"val/AP50/{k}": v for k, v in per_class.items()},
        **{f"val/IoU/{k}": v for k, v in iou_per_class.items()},
    })
    key = (mAP, -va)
    if best is None or key > best:
        best = key
        model.save_pretrained(OUT_DIR)  # best checkpoint
    model.save_pretrained(OUT_DIR / "last")  # final weights, for comparison

run.summary["best_val_mAP50"] = best[0]
plt.plot(history["train"], label="train"); plt.plot(history["val"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()

plt.plot(history["map"], label="mAP@0.5", color="green"); plt.plot(history["miou"], label="mIoU", color="orange")
plt.xlabel("epoch"); plt.ylabel("val metric"); plt.legend(); plt.show()
print("per-class AP@0.5 (last epoch):", {k: round(v, 3) for k, v in per_class.items()})
print("per-class IoU (last epoch):", {k: round(v, 3) for k, v in iou_per_class.items()})

# Log the best checkpoint (highest val mAP@0.5) as a W&B model artifact
best_artifact = wandb.Artifact(
    name=f"rt-detrv2-boxing-{BACKBONE}",
    type="model",
    metadata={"best_val_mAP50": best[0], "best_val_loss": -best[1], "backbone": BACKBONE,
              "image_size": IMAGE_SIZE, "epochs": EPOCHS},
)
best_artifact.add_file(str(OUT_DIR / "model.safetensors"))
best_artifact.add_file(str(OUT_DIR / "config.json"))
run.log_artifact(best_artifact, aliases=["best"])

run.finish()

## Inspect predictions

In [ ]:
from boxing.model import BoxingDetector

best_model = BoxingDetector.from_pretrained(OUT_DIR).to(device).eval()


def show(ds, idxs, threshold=DISPLAY_THRESHOLD, cols=3):
    rows = -(-len(idxs) // cols)  # ceil division: an n x 3 grid
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
    axes = axes.ravel()
    for ax in axes[len(idxs):]:
        ax.axis("off")
    for ax, i in zip(axes, idxs):
        x, t = ds[i]
        with torch.no_grad():
            out = best_model(pixel_values=x[None].to(device))
        probs = out.logits[0].sigmoid().cpu()
        score, cls = probs.max(-1)
        keep = score > threshold
        ax.imshow((x * STD + MEAN).permute(1, 2, 0).clamp(0, 1))
        for (cx, cy, w, h), c, s in zip(out.pred_boxes[0].cpu()[keep], cls[keep], score[keep]):
            ax.add_patch(plt.Rectangle(((cx - w / 2) * IMAGE_SIZE, (cy - h / 2) * IMAGE_SIZE), w * IMAGE_SIZE, h * IMAGE_SIZE, fill=False, color="red"))
            ax.text((cx - w / 2) * IMAGE_SIZE, (cy - h / 2) * IMAGE_SIZE, f"{id2label[int(c)]} {s:.2f}", color="red", fontsize=7)
        for (cx, cy, w, h) in t["boxes"]:
            ax.add_patch(plt.Rectangle(((cx - w / 2) * IMAGE_SIZE, (cy - h / 2) * IMAGE_SIZE), w * IMAGE_SIZE, h * IMAGE_SIZE, fill=False, color="lime"))
        ax.axis("off")
    plt.show()


N_SAMPLES = 9  # grid of 3 columns
show(val_ds, random.sample(range(len(val_ds)), N_SAMPLES))  # green = ground truth, red = prediction

In [ ]:
# Diagnostic: how confident is the model? Lower scores mean the threshold or training length is the problem.
diag_idxs = random.sample(range(len(val_ds)), 20)
best_scores, counts = [], {t: 0 for t in (0.05, 0.1, 0.2, 0.3, 0.4)}
for i in diag_idxs:
    x, _ = val_ds[i]
    with torch.no_grad():
        logits = best_model(pixel_values=x[None].to(device)).logits[0].sigmoid().cpu()
    score = logits.max(-1).values  # best class score per query
    best_scores.append(score.max().item())
    for t in counts:
        counts[t] += int((score > t).sum())

best_scores = np.array(best_scores)
print(f"top score per frame: min {best_scores.min():.3f}  median {np.median(best_scores):.3f}  max {best_scores.max():.3f}")
print("queries above threshold, summed over 20 frames:")
for t, n in counts.items():
    print(f"  > {t:.2f}: {n}")
print("final train loss:", f"{history['train'][-1]:.4f}" if history["train"] else "n/a",
      "| final val loss:", f"{history['val'][-1]:.4f}" if history["val"] else "n/a")

# mAP at IoU 0.5, 0.75 and 0.9 on the full validation set, using the best checkpoint
from torch.utils.data import DataLoader

val_eval_dl = DataLoader(val_ds, BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)
for thr in (0.5, 0.75, 0.9):
    m, per_class = mean_ap(val_eval_dl, iou_thresh=thr, net=best_model)
    print(f"mAP@{thr:.2f}: {m:.4f}  per class: {({k: round(v, 3) for k, v in per_class.items()})}")

m_iou, iou_per_class = mean_iou(val_eval_dl, net=best_model)
print(f"mIoU (score > {DISPLAY_THRESHOLD}): {m_iou:.4f}  per class: {({k: round(v, 3) for k, v in iou_per_class.items()})}")